# TP 2 — Apprentissage supervisé (MLS)
### Partie II de la conférence · Licence 1 → Master 2

> **TechForge Club · FST — Université de Nouakchott**  
> Conférence **« L’intelligence artificielle : au-delà de la théorie »** · الذكاء الاصطناعي : ما وراء النظرية  
> Animée par **Dr El Benany Med Mahmoud** · samedi 14 novembre 2026 · 12 h 30 · Amphi 5 – FST

**Trois parcours dans le même notebook — choisissez le vôtre :**

| Repère | Public conseillé | Ce que l’on vous demande |
|:---:|---|---|
| 🟢 **Découverte** | Licence 1 – Licence 2 | Exécuter, observer, modifier une valeur, répondre en une phrase. Aucune démonstration exigée. |
| 🟡 **Approfondissement** | Licence 3 | Refaire les calculs à la main, compléter du code, interpréter les résultats. |
| 🔴 **Expert** | Master 1 – Master 2 | Démontrer, implémenter sans bibliothèque, critiquer le protocole, mener un mini-projet. |

Chaque exercice **À faire** porte un repère. Un étudiant de licence peut s’arrêter aux 🟢 d’une section et passer à la suivante ; un étudiant de master traite les trois niveaux. Les cellules `verifier(...)` contrôlent votre réponse sans afficher la solution.

**Objectifs.** Maîtriser le vocabulaire données / features / cible, la régression et la classification, la séparation apprentissage / test, les métriques, l’optimisation par descente de gradient, la régularisation, la généralisation, la validation croisée et l’interprétation prudente des résultats.

**Diapositives associées :** Partie II · 01 à 18.

**Prérequis.** 🟢 exécuter une cellule, lire un graphique ; 🟡 Python, moyenne et variance ; 🔴 dérivées partielles, algèbre linéaire, scikit-learn.

**Données.** Toutes les données scolaires sont **synthétiques** : aucune décision réelle sur un étudiant ne doit être fondée sur ces modèles pédagogiques. *California Housing* est téléchargé par scikit-learn à la première utilisation ; sans connexion, un jeu synthétique de secours, clairement identifié, prend le relais.

| Section | Thème | Diapos | 🟢 | 🟡 | 🔴 | Durée |
|---|---|:---:|:---:|:---:|:---:|:---:|
| 1–2 | Vocabulaire ; programme classique vs MLS ; moindres carrés | 01–04 | ✔ | ✔ | ✔ | 25 min |
| 3 | Erreurs de régression : MAE, MSE, RMSE, R² | 06–07 | ✔ | ✔ | ✔ | 20 min |
| 4 | Descente de gradient | 12 | ✔ | ✔ | ✔ | 25 min |
| 5 | Classification, matrice de confusion, seuil | 05, 08–10 | ✔ | ✔ | ✔ | 35 min |
| 6 | k-NN, arbres, régularisation | 14–16 |  | ✔ | ✔ | 25 min |
| 7 | Biais, variance et complexité | 13 | ✔ | ✔ | ✔ | 15 min |
| 8 | Validation croisée et fuite de données | 11, 17 |  | ✔ | ✔ | 25 min |
| 9 | Étude de cas : California Housing | — |  | ✔ | ✔ | 30 min |
| 10–12 | Défis et mini-projets, barème | 18 | ✔ | ✔ | ✔ | projet |

In [ ]:
# Installation (à décommenter une seule fois si nécessaire) :
# %pip install numpy pandas matplotlib scikit-learn ipywidgets
# --- Outils communs aux quatre TP de la conférence ---
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': .3})

def verifier(nom, reponse, attendu, tol=1e-2):
    """Compare votre réponse à la valeur attendue, sans l'afficher."""
    if reponse is None:
        print(f'⏳ {nom} : à compléter (remplacez None par votre réponse)'); return
    ok = np.allclose(np.asarray(reponse, dtype=float), np.asarray(attendu, dtype=float), atol=tol)
    print(f'✅ {nom} : correct' if ok else f'❌ {nom} : à revoir — reprenez le calcul')

def explorer(f, **curseurs):
    """Curseurs interactifs si ipywidgets est installé ; sinon exécution avec les valeurs par défaut.
    Chaque curseur est donné sous la forme nom=(min, max, pas, valeur_par_defaut)."""
    try:
        import ipywidgets as w
        ctl = {k: (w.FloatSlider if any(isinstance(x, float) for x in v) else w.IntSlider)(
                   min=v[0], max=v[1], step=v[2], value=v[3], description=k, continuous_update=False)
               for k, v in curseurs.items()}
        w.interact(f, **ctl)
    except ImportError:
        print('(ipywidgets absent : résultat pour les valeurs par défaut — changez-les dans l’appel pour explorer)')
        f(**{k: v[3] for k, v in curseurs.items()})
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.datasets import load_breast_cancer, load_diabetes
print('Environnement prêt. Graine =', SEED)

## 1 — Vocabulaire et modèle mathématique  *(diapos 01, 11)*

Une **observation** est une ligne ; une **feature** est une variable explicative ; la **cible** $y$ est la réponse à prédire. $X\in\mathbb{R}^{n\times p}$ contient $n$ observations et $p$ features. Le modèle $f_\theta$ apprend ses paramètres $\theta$ en minimisant une fonction de coût sur l’ensemble d’entraînement.

- **Régression :** $\hat y\in\mathbb R$ (prix, température, note).
- **Classification binaire :** $\hat y\in\{0,1\}$ (réussite / non-réussite).
- **Généralisation :** performance sur des observations jamais utilisées pour entraîner le modèle.

> L’apprentissage supervisé apprend une relation **statistique**, pas nécessairement une loi causale.

In [ ]:
exemple = pd.DataFrame({'Distance_km': [1, 2, 3, 4], 'Heure_pointe': [0, 1, 0, 1], 'Prix': [7, 10, 11, 14]})
display(exemple)
X_ex, y_ex = exemple[['Distance_km', 'Heure_pointe']], exemple['Prix']
print('X :', X_ex.shape, '→ n =', X_ex.shape[0], 'observations, p =', X_ex.shape[1], 'features | cible :', y_ex.name)

**À faire** — 🟢 Dans ce tableau, quelle colonne est la cible ? Est-ce un problème de régression ou de classification ? Que deviendrait-il si la cible était « course chère : oui / non » ?

## 2 — Programme classique et apprentissage supervisé  *(diapos 03–04)*

**Programme classique :** données $x$ + règle fournie $f$ → résultat $f(x)$.

**MLS :** exemples $(x_i,y_i)$ + algorithme d’apprentissage → modèle $f_{\hat\theta}$ ; puis nouvelle entrée $x_\ast$ → prédiction $\hat y_\ast$.

In [ ]:
def prix_programme_classique(distance):
    return 2 * distance + 5

x_train = np.array([1, 2, 3, 4]).reshape(-1, 1)
y_train = np.array([7, 9, 11, 13])
modele = LinearRegression().fit(x_train, y_train)
print('Règle explicite (6 km) :', prix_programme_classique(6))
print(f'Règle apprise : prix ≈ {modele.coef_[0]:.2f} × distance + {modele.intercept_:.2f}')
print('Prédiction MLS (6 km) :', round(modele.predict([[6]])[0], 2))
plt.scatter(x_train[:, 0], y_train, label='Exemples observés')
xx = np.linspace(0, 7, 100); plt.plot(xx, modele.predict(xx.reshape(-1, 1)), label='Droite apprise')
plt.axvspan(4, 7, alpha=.08, color='red', label='Extrapolation')
plt.xlabel('Distance (km)'); plt.ylabel('Prix (unités fictives)'); plt.legend(); plt.show()

### Calcul à la main : moindres carrés

Pour $\hat y_i=ax_i+b$, on minimise $J(a,b)=\frac1n\sum_i(y_i-ax_i-b)^2$ :

$$a=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2},\qquad b=\bar y-a\bar x.$$

Avec $(x,y)=\{(1,7),(2,9),(3,11),(4,13)\}$ : $\bar x=2{,}5$, $\bar y=10$, $\sum(x_i-\bar x)(y_i-\bar y)=10$, $\sum(x_i-\bar x)^2=5$, donc $a=2$, $b=5$.

**À faire**

- 🟢 Prédisez à la main le prix d’une course de 10 km. Cette prédiction est-elle aussi fiable que celle pour 3 km ?
- 🟡 Reproduisez les quatre sommes sur papier pour les points $(1,3),(2,5),(3,8)$, puis vérifiez.
- 🔴 Démontrez les deux formules en annulant $\partial J/\partial a$ et $\partial J/\partial b$.

In [ ]:
x = x_train[:, 0]
a = np.sum((x - x.mean()) * (y_train - y_train.mean())) / np.sum((x - x.mean()) ** 2)
b0 = y_train.mean() - a * x.mean()
print('Calcul manuel vérifié : a =', a, ', b =', b0)

ma_pente, mon_ordonnee = None, None      # 🟡 pour les points (1,3), (2,5), (3,8)
ref = LinearRegression().fit([[1], [2], [3]], [3, 5, 8])
verifier('pente a', ma_pente, ref.coef_[0]); verifier('ordonnée b', mon_ordonnee, ref.intercept_)

## 3 — Mesurer les erreurs de régression  *(diapos 06–07)*

Résidu $e_i=y_i-\hat y_i$.

$$\mathrm{MAE}=\frac1n\sum|e_i|,\quad \mathrm{MSE}=\frac1n\sum e_i^2,\quad \mathrm{RMSE}=\sqrt{\mathrm{MSE}},\quad R^2=1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}.$$

MAE et RMSE ont l’unité de la cible ; la MSE a son unité au carré. $R^2$ peut être **négatif** : le modèle fait alors moins bien que la prédiction constante $\bar y$.

**Exemple du cours.** $y=(7,9,11)$, $\hat y=(8,8,8)$ : MAE $=5/3\approx1{,}67$ ; MSE $=11/3\approx3{,}67$ ; RMSE $\approx1{,}915$ ; $SS_{tot}=8$, $SS_{res}=11$, donc $R^2=1-11/8=-0{,}375$.

In [ ]:
reel = np.array([7, 9, 11]); predit = np.array([8, 8, 8]); err = reel - predit
display(pd.DataFrame({'réel': reel, 'prédit': predit, 'résidu': err, '|résidu|': abs(err), 'résidu²': err ** 2}))
print('MAE =', round(abs(err).mean(), 3), '| MSE =', round(np.mean(err ** 2), 3), '| RMSE =', round(np.sqrt(np.mean(err ** 2)), 3), '| R² =', r2_score(reel, predit))
assert np.isclose(abs(err).mean(), 5 / 3) and np.isclose(np.mean(err ** 2), 11 / 3) and np.isclose(r2_score(reel, predit), -0.375)

In [ ]:
def explorer_erreurs(derniere_prediction=8):
    yp = np.array([8, 8, derniere_prediction]); e = reel - yp
    print(f'MAE = {np.mean(abs(e)):.3f} | MSE = {np.mean(e ** 2):.3f} | RMSE = {np.sqrt(np.mean(e ** 2)):.3f} | R² = {r2_score(reel, yp):.3f}')
    plt.bar(np.arange(3) + 1, e); plt.axhline(0, color='black'); plt.xlabel('Observation'); plt.ylabel('Résidu'); plt.show()

explorer(explorer_erreurs, derniere_prediction=(0, 16, 1, 8))

**À faire**

- 🟢 Remplacez la dernière prédiction 8 par 4, puis par 11. Quelle métrique réagit le plus à une grande erreur ?
- 🟡 Pour $y=(2,4,6,8)$ et $\hat y=(3,5,7,9)$, calculez à la main MAE, RMSE et $R^2$.
- 🔴 Montrez que la constante $c$ qui minimise la MSE est la moyenne de $y$, et que celle qui minimise la MAE est une médiane. Qu’en déduire sur la sensibilité aux valeurs aberrantes ?

In [ ]:
ma_mae, ma_rmse, mon_r2 = None, None, None     # 🟡
yv, yh = np.array([2, 4, 6, 8]), np.array([3, 5, 7, 9])
verifier('MAE', ma_mae, mean_absolute_error(yv, yh)); verifier('RMSE', ma_rmse, np.sqrt(mean_squared_error(yv, yh))); verifier('R²', mon_r2, r2_score(yv, yh))

## 4 — Descente de gradient : modèle linéaire à une variable  *(diapo 12)*

🟢 **L’idée.** On part de paramètres quelconques et on les corrige pas à pas dans la direction qui fait baisser le coût. Le **taux d’apprentissage** $\eta$ règle la taille des pas : trop petit, c’est lent ; trop grand, on dépasse le minimum.

🟡 **Les formules.** Pour $J(a,b)=\dfrac1{2n}\sum_{i=1}^n(ax_i+b-y_i)^2$ :

$$\frac{\partial J}{\partial a}=\frac1n\sum_i(ax_i+b-y_i)\,x_i,\qquad \frac{\partial J}{\partial b}=\frac1n\sum_i(ax_i+b-y_i),$$

$$a\leftarrow a-\eta\,\frac{\partial J}{\partial a},\qquad b\leftarrow b-\eta\,\frac{\partial J}{\partial b}.$$

🔴 **La démonstration.** Par linéarité, $\dfrac{\partial J}{\partial a}=\dfrac1{2n}\sum_i\dfrac{\partial}{\partial a}(ax_i+b-y_i)^2$. La règle de dérivation des fonctions composées donne $\dfrac{\partial}{\partial a}u^2=2u\,\dfrac{\partial u}{\partial a}$ avec $u=ax_i+b-y_i$ et $\dfrac{\partial u}{\partial a}=x_i$ ($b$ et $y_i$ sont constants). D’où $\dfrac{\partial J}{\partial a}=\dfrac1{2n}\sum_i2(ax_i+b-y_i)x_i=\dfrac1n\sum_i(ax_i+b-y_i)x_i$. Le même calcul avec $\dfrac{\partial u}{\partial b}=1$ donne $\dfrac{\partial J}{\partial b}$. Le facteur $\tfrac12$ du coût ne sert qu’à simplifier ce $2$.

In [ ]:
def descente_gradient(x, y, eta=.03, epochs=600):
    a, b, historique = 0., 0., []
    for t in range(epochs):
        erreur = a * x + b - y
        historique.append(np.mean(erreur ** 2) / 2)
        a -= eta * np.mean(erreur * x)
        b -= eta * np.mean(erreur)
    return a, b, np.array(historique)

for eta in [.005, .03, .1]:
    aa, bb, h = descente_gradient(x, y_train, eta)
    plt.plot(h, label=f'η = {eta} ; a = {aa:.2f}, b = {bb:.2f}')
plt.yscale('log'); plt.xlabel('Itération'); plt.ylabel('Coût J'); plt.legend(); plt.title('Trois taux d’apprentissage'); plt.show()

# Un pas trop grand : le coût augmente au lieu de diminuer
_, _, h_div = descente_gradient(x, y_train, eta=.25, epochs=12)
print('η = 0,25 — coût aux 6 premières itérations :', np.round(h_div[:6], 1), '→ divergence')

**À faire**

- 🟢 Quel taux converge le plus vite parmi 0,005 ; 0,03 ; 0,1 ? Que se passe-t-il avec 0,25 ?
- 🟡 Calculez à la main le **premier pas** avec $a=b=0$, $\eta=0{,}03$ et les données $(1,7),(2,9),(3,11),(4,13)$, puis comparez au code.
- 🔴 La matrice hessienne de $J$ est $H=\begin{pmatrix}\overline{x^2}&\bar x\\\bar x&1\end{pmatrix}$. Calculez sa plus grande valeur propre $\lambda_{\max}$ et vérifiez que la descente diverge dès que $\eta>2/\lambda_{\max}$.

In [ ]:
mon_a1, mon_b1 = None, None        # 🟡 valeurs de a et b après le premier pas
a1 = 0 - .03 * np.mean((0 * x + 0 - y_train) * x); b1 = 0 - .03 * np.mean(0 * x + 0 - y_train)
verifier('a après un pas', mon_a1, a1, tol=1e-3); verifier('b après un pas', mon_b1, b1, tol=1e-3)

## 5 — Réussite scolaire : classification binaire (données synthétiques)  *(diapos 05, 08–10)*

On simule des notes de **mathématiques, programmation, statistiques et anglais**, des heures d’étude, et une cible **réussite** fabriquée par une règle fictive et du bruit. Elle ne représente **ni des étudiants réels ni une politique universitaire**.

Le modèle logistique estime $p(y=1\mid x)=\sigma(w^\top x+b)$, avec $\sigma(z)=1/(1+e^{-z})$. La classe prédite dépend d’un seuil $t$ : $\hat y=1$ si $p\ge t$.

In [ ]:
g = np.random.default_rng(SEED)
n_et = 240
notes = g.integers(4, 21, size=(n_et, 4)); heures = g.integers(0, 16, size=n_et)
score = .34 * notes[:, 0] + .30 * notes[:, 1] + .22 * notes[:, 2] + .14 * notes[:, 3] + .16 * heures + g.normal(0, 2.2, n_et)
etudiants = pd.DataFrame(notes, columns=['Mathématiques', 'Programmation', 'Statistiques', 'Anglais'])
etudiants['Heures_etude'] = heures; etudiants['Reussite'] = (score >= 12).astype(int)
display(etudiants.head(8)); print(etudiants['Reussite'].value_counts().to_dict())

X = etudiants.drop(columns='Reussite'); y = etudiants['Reussite']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=SEED, stratify=y)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]; yp = (proba >= .5).astype(int)
base = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
print(f'Référence « classe majoritaire » : accuracy = {base.score(Xte, yte):.3f} | modèle logistique : {accuracy_score(yte, yp):.3f}')
print(classification_report(yte, yp, zero_division=0))
ConfusionMatrixDisplay.from_predictions(yte, yp, display_labels=['Non réussite', 'Réussite'], cmap='Blues')
plt.title('Matrice de confusion — jeu de test'); plt.grid(False); plt.show()

### Matrice de confusion et métriques

Positif = « réussite ». $VP$ : réussite correctement prédite ; $FP$ : réussite prédite à tort ; $FN$ : réussite manquée ; $VN$ : non-réussite correctement prédite.

$$\text{Accuracy}=\frac{VP+VN}{N},\quad \text{Précision}=\frac{VP}{VP+FP},\quad \text{Rappel}=\frac{VP}{VP+FN},\quad F_1=\frac{2\,VP}{2\,VP+FP+FN}.$$

**Exemple du cours.** $VP=40$, $FN=10$, $FP=5$, $VN=45$ : accuracy $=85\,\%$ ; précision $=40/45\approx0{,}889$ ; rappel $=40/50=0{,}8$ ; $F_1=80/95\approx0{,}842$.

**À faire**

- 🟢 Dans la matrice affichée, combien d’étudiants le modèle a-t-il déclarés « réussite » à tort ?
- 🟡 Pour $VP=18$, $VN=14$, $FP=4$, $FN=6$, calculez à la main accuracy, précision, rappel et $F_1$.
- 🔴 Sur 1 000 messages dont 10 spams, un modèle répond toujours « normal ». Calculez son accuracy et son rappel, puis expliquez pourquoi la comparaison à la **référence majoritaire** est indispensable.

In [ ]:
ma_accuracy, ma_precision, mon_rappel, mon_f1 = None, None, None, None     # 🟡
VP, VN, FP, FN = 18, 14, 4, 6
verifier('Accuracy', ma_accuracy, (VP + VN) / (VP + VN + FP + FN)); verifier('Précision', ma_precision, VP / (VP + FP))
verifier('Rappel', mon_rappel, VP / (VP + FN)); verifier('F1', mon_f1, 2 * VP / (2 * VP + FP + FN))

### Étudier le seuil et les compromis

Modifier le seuil change les décisions et les métriques, **pas les probabilités prédites**. Le meilleur seuil dépend des coûts respectifs des faux positifs et des faux négatifs.

In [ ]:
def metriques_seuil(t=0.5):
    pred = (proba >= t).astype(int)
    print(f'Seuil {t:.2f} | accuracy {accuracy_score(yte, pred):.3f} | précision {precision_score(yte, pred, zero_division=0):.3f} | '
          f'rappel {recall_score(yte, pred, zero_division=0):.3f} | F1 {f1_score(yte, pred, zero_division=0):.3f}')
    ConfusionMatrixDisplay.from_predictions(yte, pred, cmap='Blues'); plt.grid(False); plt.show()

explorer(metriques_seuil, t=(0.05, 0.95, 0.05, 0.5))

seuils = np.arange(.05, .96, .05)
plt.plot(seuils, [precision_score(yte, proba >= t, zero_division=0) for t in seuils], 'o-', label='Précision')
plt.plot(seuils, [recall_score(yte, proba >= t, zero_division=0) for t in seuils], 's-', label='Rappel')
plt.xlabel('Seuil'); plt.legend(); plt.title('Compromis précision / rappel'); plt.show()

### Simuler un nouvel étudiant (exemple purement fictif)

La valeur affichée est un **score du modèle**, dépendant du jeu synthétique et de ses hypothèses. Ce n’est pas une probabilité fiable de réussite réelle.

In [ ]:
nouvel_etudiant = pd.DataFrame([{'Mathématiques': 14, 'Programmation': 16, 'Statistiques': 13, 'Anglais': 12, 'Heures_etude': 8}])
p = clf.predict_proba(nouvel_etudiant)[0, 1]
print(f'Score de réussite prédit (simulation) : {p:.1%} ; classe au seuil 0,5 : {int(p >= .5)}')

**À faire**

- 🟢 Baissez les notes du nouvel étudiant jusqu’à faire basculer la prédiction. Quelle matière pèse le plus ?
- 🟡 Si l’on veut repérer tôt les étudiants **en difficulté** pour les accompagner, vaut-il mieux un seuil haut ou bas ? Quelle erreur est la plus coûteuse ?
- 🔴 Affichez les coefficients du modèle (`clf[-1].coef_`). Sont-ils comparables entre features ? Pourquoi la standardisation est-elle nécessaire pour les comparer ? Peut-on en déduire une causalité ?

## 6 — k-NN, arbres et régularisation  *(diapos 14–16)*

- **k-NN :** $d(x,z)=\sqrt{\sum_j(x_j-z_j)^2}$, puis vote parmi les $k$ voisins les plus proches ; mettre les variables à la même échelle.
- **Arbre :** partitions successives de l’espace des features. Impureté de Gini $G=1-\sum_c p_c^2$ ; pour 3 A et 1 B, $G=1-(3/4)^2-(1/4)^2=0{,}375$.
- **Régularisation L2 :** ajouter $\lambda\|w\|_2^2$ à l’objectif (dans scikit-learn, `C` $=1/\lambda$ pour la régression logistique).
- **Surapprentissage :** très bonne performance d’entraînement, performance de test dégradée.

In [ ]:
for nom, model in [('k-NN (k = 5)', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))),
                   ('Arbre, profondeur 3', DecisionTreeClassifier(max_depth=3, random_state=SEED)),
                   ('Arbre sans limite', DecisionTreeClassifier(random_state=SEED))]:
    model.fit(Xtr, ytr)
    print(f'{nom:22s} entraînement = {model.score(Xtr, ytr):.3f} | test = {model.score(Xte, yte):.3f}')
arbre = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xtr, ytr)
plt.figure(figsize=(15, 6)); plot_tree(arbre, feature_names=list(X.columns), class_names=['Non', 'Oui'], filled=True, rounded=True, fontsize=8); plt.show()

In [ ]:
# Effet de k (k-NN) et de la régularisation (régression logistique)
ks = [1, 3, 5, 9, 15, 25, 41]
sc = [(m.score(Xtr, ytr), m.score(Xte, yte)) for m in (make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(Xtr, ytr) for k in ks)]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ks, [s[0] for s in sc], 'o-', label='entraînement'); ax[0].plot(ks, [s[1] for s in sc], 's-', label='test')
ax[0].set_xlabel('k'); ax[0].set_ylabel('accuracy'); ax[0].set_title('k-NN : k petit = frontière instable'); ax[0].legend()
Cs = [1e-3, 1e-2, 1e-1, 1, 10, 100]
normes = [np.linalg.norm(make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000)).fit(Xtr, ytr)[-1].coef_) for C in Cs]
ax[1].semilogx(Cs, normes, 'o-'); ax[1].set_xlabel('C = 1/λ'); ax[1].set_ylabel('‖w‖'); ax[1].set_title('Plus λ est grand, plus les coefficients rétrécissent')
plt.tight_layout(); plt.show()

**À faire**

- 🟡 L’arbre sans limite atteint 100 % à l’entraînement. Est-ce une bonne nouvelle ? Comparez à son score de test.
- 🟡 Calculez à la main l’indice de Gini d’un nœud contenant 5 A et 5 B, puis d’un nœud contenant 10 A.
- 🔴 Implémentez k-NN **sans scikit-learn** (distances, tri, vote) et retrouvez les prédictions de `KNeighborsClassifier` sur le jeu de test standardisé.

In [ ]:
mon_gini_5_5, mon_gini_10_0 = None, None       # 🟡
verifier('Gini (5 A, 5 B)', mon_gini_5_5, 1 - .5**2 - .5**2); verifier('Gini (10 A)', mon_gini_10_0, 0)

## 7 — Biais, variance et complexité  *(diapo 13)*

On ajuste des polynômes de degrés 1, 3 et 15 sur 30 points bruités issus de $\sin(2x)$. Une courbe très flexible peut suivre le bruit de l’entraînement sans généraliser.

In [ ]:
g = np.random.default_rng(SEED)
x_all = np.sort(g.uniform(-2, 2, 30)); y_all = np.sin(2 * x_all) + g.normal(0, .25, 30)     # peu de données : terrain favorable au surapprentissage
ixtr, ixte = train_test_split(np.arange(len(x_all)), test_size=.3, random_state=SEED)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(x_all[ixtr], y_all[ixtr], label='Entraînement'); ax.scatter(x_all[ixte], y_all[ixte], label='Test', marker='x')
grille = np.linspace(-2, 2, 200).reshape(-1, 1); lignes = []
for deg in [1, 3, 15]:
    m = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression()).fit(x_all[ixtr, None], y_all[ixtr])
    rmse = lambda idx: np.sqrt(mean_squared_error(y_all[idx], m.predict(x_all[idx, None])))
    lignes.append({'degré': deg, 'RMSE entraînement': round(rmse(ixtr), 3), 'RMSE test': round(rmse(ixte), 3)})
    ax.plot(grille, m.predict(grille), label=f'Degré {deg}')
ax.set_ylim(-2, 2); ax.legend(); plt.show(); display(pd.DataFrame(lignes))

**À faire**

- 🟢 Quel degré décrit le mieux la tendance générale ? Lequel « suit le bruit » ?
- 🟡 Tracez les RMSE d’entraînement et de test pour tous les degrés de 1 à 15. Où se trouve le minimum de l’erreur de test ?
- 🔴 Remplacez `LinearRegression()` par `Ridge(alpha=...)` au degré 15 et faites varier `alpha`. Reliez le résultat à la décomposition erreur = biais² + variance + bruit.

## 8 — Validation croisée et fuite de données  *(diapos 11, 17)*

**Bon protocole :** conserver un test final non touché ; choisir les hyperparamètres uniquement sur l’entraînement (validation croisée). Placer l’imputation, la standardisation et toute transformation apprise **dans un Pipeline**.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), Xtr, ytr, cv=cv, scoring='f1')
print('F1 par pli :', np.round(scores, 3), '| moyenne :', round(scores.mean(), 3), '| écart-type :', round(scores.std(), 3))

### Démonstration : une fuite de données fabrique de la performance

Expérience : 100 observations, 2 000 features **purement aléatoires**, étiquettes tirées à pile ou face. Aucun modèle ne peut faire mieux que 50 %.

- **Protocole fautif :** on sélectionne les 20 features les plus liées à $y$ sur **toutes** les données, puis on fait la validation croisée.
- **Protocole correct :** la sélection est dans le Pipeline, donc refaite sur chaque pli d’entraînement.

In [ ]:
g = np.random.default_rng(SEED)
X_bruit = g.normal(size=(100, 2000)); y_bruit = g.integers(0, 2, 100)
X_fuite = SelectKBest(f_classif, k=20).fit_transform(X_bruit, y_bruit)                    # fuite : y du futur pli de test déjà utilisé
fautif = cross_val_score(LogisticRegression(max_iter=1000), X_fuite, y_bruit, cv=5).mean()
correct = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)), X_bruit, y_bruit, cv=5).mean()
print(f'Accuracy « avec fuite » : {fautif:.2f}  ← illusion | accuracy avec Pipeline : {correct:.2f}  ← proche du hasard, comme attendu')

**À faire**

- 🟡 Expliquez en deux phrases pourquoi le protocole fautif obtient un score élevé sur du bruit.
- 🟡 Citez deux autres fuites courantes (indice : standardisation, doublons, variable connue seulement après l’événement).
- 🔴 Choisissez `k` de k-NN par validation croisée sur `Xtr` (grille 1 à 41), puis évaluez **une seule fois** sur `Xte`. Comparez au `k` qui maximise directement le score de test : pourquoi ce second choix est-il biaisé ?

## 9 — Étude de cas réelle : California Housing

Jeu **California Housing** (recensement californien de 1990) : 20 640 zones, 8 features, cible `MedHouseVal` = valeur médiane des logements par zone, en unités de **100 000 dollars**. Ce ne sont pas des prix individuels ; le jeu est historique et agrégé, et ne doit pas servir à fixer des prix actuels.

**Modèle :** $\hat y=\beta_0+\sum_{j=1}^8\beta_jx_j$, coefficients estimés en minimisant $\sum_i(y_i-\hat y_i)^2$. La solution $\hat\beta=(X^\top X)^{-1}X^\top y$ suppose $X^\top X$ inversible et n’est pas calculée par inversion explicite en pratique. La régression Ridge ajoute la pénalité $\lambda\|\beta\|_2^2$.

In [ ]:
from sklearn.datasets import fetch_california_housing
try:
    df_housing = fetch_california_housing(as_frame=True).frame.copy()
    housing_source = 'CALIFORNIA HOUSING RÉEL (1990)'
except Exception as exc:
    print('Téléchargement indisponible :', str(exc)[:120])
    print('ATTENTION : utilisation d’un jeu SYNTHÉTIQUE de secours, pas California Housing.')
    g = np.random.default_rng(SEED); n_h = 3000
    sim = pd.DataFrame({'MedInc': g.uniform(.5, 12, n_h), 'HouseAge': g.uniform(1, 50, n_h), 'AveRooms': g.uniform(2, 9, n_h),
                        'AveBedrms': g.uniform(.5, 2.5, n_h), 'Population': g.integers(50, 4000, n_h), 'AveOccup': g.uniform(1, 6, n_h),
                        'Latitude': g.uniform(32, 42, n_h), 'Longitude': g.uniform(-124, -114, n_h)})
    sim['MedHouseVal'] = np.clip(.35 + .34 * sim.MedInc + .012 * sim.HouseAge - .08 * sim.AveBedrms + g.normal(0, .55, n_h), .15, 5)
    df_housing = sim; housing_source = 'DONNÉES SYNTHÉTIQUES DE SECOURS'
print('SOURCE :', housing_source, '| forme :', df_housing.shape)
display(df_housing.describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_housing['MedHouseVal'], bins=35); axes[0].set_xlabel('Valeur médiane (× 100 000 $)'); axes[0].set_ylabel('Fréquence')
axes[1].scatter(df_housing['MedInc'], df_housing['MedHouseVal'], s=3, alpha=.25)
axes[1].set_xlabel('Revenu médian (MedInc)'); axes[1].set_ylabel('Valeur médiane (× 100 000 $)'); plt.tight_layout(); plt.show()
print(df_housing.corr(numeric_only=True)['MedHouseVal'].sort_values(ascending=False).round(3))

### Calcul manuel d’une prédiction et de son résidu

Le modèle standardisé utilise $z_j=(x_j-\mu_j)/s_j$ et prédit $\hat y=\beta_0+\sum_j\beta_jz_j$. Le tableau expose chaque contribution. Il faut appliquer la **même** standardisation qu’à l’entraînement.

In [ ]:
Xh = df_housing.drop(columns='MedHouseVal'); yh = df_housing['MedHouseVal']
Xhtr, Xhte, yhtr, yhte = train_test_split(Xh, yh, test_size=.2, random_state=SEED)
reg = make_pipeline(StandardScaler(), LinearRegression()).fit(Xhtr, yhtr); yp_h = reg.predict(Xhte)
print('SOURCE :', housing_source)
print(f'MAE = {mean_absolute_error(yhte, yp_h):.4f} | RMSE = {np.sqrt(mean_squared_error(yhte, yp_h)):.4f} | R² = {r2_score(yhte, yp_h):.4f} | MAE ≈ {mean_absolute_error(yhte, yp_h) * 100000:,.0f} $')
scaler, lr = reg[0], reg[1]
ex = Xhte.iloc[[0]]; z = scaler.transform(ex)[0]; contributions = z * lr.coef_
display(pd.DataFrame({'Feature': Xh.columns, 'Valeur brute': ex.iloc[0].values, 'Standardisée z': z, 'Coefficient β': lr.coef_, 'Contribution β×z': contributions}).round(4))
print(f'Intercept {lr.intercept_:.4f} + contributions {contributions.sum():.4f} = {lr.intercept_ + contributions.sum():.4f} | scikit-learn : {reg.predict(ex)[0]:.4f} | valeur observée : {yhte.iloc[0]:.4f}')
assert np.isclose(lr.intercept_ + contributions.sum(), reg.predict(ex)[0])

In [ ]:
# Comparaison descriptive : ne pas choisir le meilleur modèle sur le test
for nom, model in [('Référence : moyenne', None), ('Linéaire', make_pipeline(StandardScaler(), LinearRegression())),
                   ('Ridge α = 1', make_pipeline(StandardScaler(), Ridge(alpha=1))), ('Ridge α = 100', make_pipeline(StandardScaler(), Ridge(alpha=100)))]:
    pred = np.full(len(yhte), yhtr.mean()) if model is None else model.fit(Xhtr, yhtr).predict(Xhte)
    print(f'{nom:20s} MAE = {mean_absolute_error(yhte, pred):.3f} | RMSE = {np.sqrt(mean_squared_error(yhte, pred)):.3f} | R² = {r2_score(yhte, pred):.3f}')
plt.scatter(yhte, yp_h, s=5, alpha=.3); lo, hi = min(yhte.min(), yp_h.min()), max(yhte.max(), yp_h.max())
plt.plot([lo, hi], [lo, hi], 'k--'); plt.xlabel('Valeur réelle'); plt.ylabel('Valeur prédite'); plt.title('Prédictions vs observations'); plt.show()

**À faire**

- 🟡 La MAE est-elle grande ou petite au regard de la valeur médiane typique ? Comparez à la référence « moyenne ».
- 🔴 **Généralisation spatiale.** Des zones voisines se ressemblent : une séparation aléatoire peut surestimer la capacité à prédire dans une région nouvelle. Construisez une validation par groupes géographiques (par exemple `GroupKFold` sur des tranches de latitude) et comparez ses erreurs à celles du découpage aléatoire.
- 🔴 Quelles sont les limites éthiques, temporelles et statistiques d’un modèle de prix fondé sur un recensement de 1990 ?

## 10 — Défis et mini-projets (à rendre)

| Niveau | Sujet | Travail demandé |
|---|---|---|
| 🟢 L1–L2 | **Défi découverte — le prix d’une course** | Inventer 8 couples (distance, prix), ajuster une droite, prédire pour 3 nouvelles distances, calculer la MAE à la main sur 3 exemples, commenter en 5 lignes. |
| 🟡 L3 | **Défi — détecter les étudiants en difficulté** | Reprendre la section 5 avec « non-réussite » comme classe positive ; choisir un seuil en justifiant le coût des FP et des FN ; comparer à la référence majoritaire. |
| 🔴 M1–M2 | **Mini-projet A (régression)** et **mini-projet B (classification)** ci-dessous | Protocole complet, sans fuite. |

### Mini-projet A — Régression 🔴

**Jeu :** `load_diabetes()` (442 observations, 10 variables numériques ; cible = mesure quantitative de progression d’une maladie un an après le début de l’étude). Usage strictement pédagogique, **pas** de conseil médical.

**Travail demandé :** (1) décrire $X$, $y$, $n$, $p$ ; (2) analyse exploratoire ; (3) découpage 80/20 ; (4) référence : moyenne de $y$ sur l’entraînement ; (5) `LinearRegression` et `Ridge` en pipeline ; (6) choix de $\alpha$ par validation croisée **sur l’entraînement seulement** ; (7) MAE, RMSE, $R^2$ sur le test ; (8) graphique des résidus ; (9) interpréter deux coefficients avec prudence ; (10) limites et conclusion.

In [ ]:
# MINI-PROJET A — cellule de départ (compléter les TODO)
diabetes = load_diabetes(as_frame=True)
X_reg, y_reg = diabetes.data, diabetes.target
print('Dimensions :', X_reg.shape, '| cible :', y_reg.shape)
# TODO 1 : train_test_split
# TODO 2 : référence = moyenne de y sur l'entraînement
# TODO 3 : modèles + validation croisée sur l'entraînement
# TODO 4 : tableau MAE / RMSE / R² sur le test
# TODO 5 : graphique des résidus et analyse

### Mini-projet B — Classification 🔴

**Jeu :** `load_breast_cancer()` (569 observations, 30 caractéristiques numériques de cellules ; cible binaire). **Jeu pédagogique historique uniquement**, jamais un dispositif de diagnostic.

**Travail demandé :** (1) inspecter les classes (0 = *malignant*, 1 = *benign* dans ce jeu) ; (2) choisir explicitement la **classe positive** et le justifier ; (3) découpage stratifié ; (4) référence majoritaire ; (5) régression logistique, k-NN, arbre ; (6) standardisation dans le pipeline ; (7) validation croisée pour $k$, la profondeur et la régularisation ; (8) matrice de confusion, précision, rappel, $F_1$, accuracy ; (9) coût des faux négatifs selon la classe positive ; (10) effet du seuil et limites.

**Livrables (A et B) :** notebook reproductible + rapport de 2 pages + tableau comparatif + conclusion argumentée. **Ne pas utiliser le test pour choisir les hyperparamètres ; ne pas présenter les résultats comme cliniquement utilisables.**

In [ ]:
# MINI-PROJET B — cellule de départ (compléter les TODO)
cancer = load_breast_cancer(as_frame=True)
X_cls, y_cls = cancer.data, cancer.target
print('Dimensions :', X_cls.shape, '| classes :', list(cancer.target_names), '| effectifs :', y_cls.value_counts().to_dict())
# TODO 1 : définir la classe positive (par exemple malignité = 1 par recodage)
# TODO 2 : découpage entraînement / test stratifié
# TODO 3 : références + pipelines + validation croisée
# TODO 4 : matrice de confusion et métriques sur le test
# TODO 5 : effet du seuil, risques et limites

## 11 — Barème (100 points)

| Critère | Points |
|---|---:|
| Reproductibilité, structure, explications | 10 |
| Exploration des données et interprétation des variables | 15 |
| Protocole entraînement / validation / test sans fuite | 20 |
| Références, modèles et choix d’hyperparamètres | 20 |
| Calculs mathématiques et métriques interprétées | 20 |
| Visualisations, limites, éthique et conclusion | 15 |

Pour les parcours 🟢 et 🟡, le barème s’applique aux exercices du niveau choisi et au défi correspondant.

## 12 — Autoévaluation

1. Quelle différence entre une feature et la cible ?
2. Quelle différence entre classification et régression ?
3. Pourquoi ne faut-il pas entraîner — ni choisir un hyperparamètre — sur le test ?
4. Quand MAE et RMSE divergent-elles ?
5. Que signifie un $R^2$ négatif ?
6. Pourquoi l’accuracy seule peut-elle être trompeuse ?
7. Que change le seuil d’un classificateur ? Que ne change-t-il pas ?
8. Qu’est-ce qu’une fuite de données ? Donnez un exemple.
9. Qu’est-ce qu’un modèle qui généralise ?

**Transition.** Si l’on dispose seulement de $X$, sans $y$, peut-on découvrir des groupes, des représentations ou des anomalies ? C’est l’objet du **TP 3 — apprentissage non supervisé**.

**Références.** Guide utilisateur de scikit-learn (*supervised learning*, *model evaluation*, *common pitfalls*) ; documentation de `fetch_california_housing` ; James et al., *An Introduction to Statistical Learning* ; Hastie, Tibshirani, Friedman, *The Elements of Statistical Learning*.